# ARGUS — YOLO12x Finetuning on BDD100K + IDD

**Model:** YOLO12x (ultralytics 8.4.x) — flagship YOLO as of 2025  
**Datasets:** BDD100K (70k images, diverse conditions) + IDD (10k images, Indian traffic)  
**Target classes:** car · motorcycle · bus · truck · bicycle  
**Expected mAP50 (post-finetune):** ~78–82%  

Attach datasets:
- `a7madmostafa/bdd100k-yolo`
- `abhishekprajapat/idd-20k`

Enable GPU accelerator (T4 x2 recommended).

In [ ]:
# Install / upgrade ultralytics
!pip install -q ultralytics>=8.4.0
import ultralytics
ultralytics.checks()

In [ ]:
import os, glob

# ── Explore dataset paths ─────────────────────────────────────────────────────
INPUT = '/kaggle/input'
for root, dirs, files in os.walk(INPUT):
    depth = root.replace(INPUT, '').count(os.sep)
    if depth > 3:
        continue
    indent = '  ' * depth
    print(f'{indent}{os.path.basename(root)}/')
    if depth == 3:
        sample = files[:4]
        for f in sample:
            print(f'{indent}  {f}')
        if len(files) > 4:
            print(f'{indent}  ... ({len(files)} files total)')

In [ ]:
import json, shutil, xml.etree.ElementTree as ET
import cv2, numpy as np
from pathlib import Path

# ── Configuration ─────────────────────────────────────────────────────────────
# Adjust these if explore cell shows different paths
BDD100K_DIR    = '/kaggle/input/bdd100k-yolo'
IDD_DIR        = '/kaggle/input/idd-20k/IDD_Detection'

OUT_DIR        = Path('/kaggle/working/argus_dataset')
MODEL_OUT      = Path('/kaggle/working/models')

EPOCHS      = 50
BATCH       = 16     # T4 x2: safe at imgsz=640; use 8 for single T4
IMGSZ       = 640
WORKERS     = 4
MODEL       = 'yolo12x.pt'

CLASS_NAMES = ['car', 'motorcycle', 'bus', 'truck', 'bicycle']

BDD_MAP = {'car': 0, 'motorcycle': 1, 'bus': 2, 'truck': 3, 'bicycle': 4}
IDD_MAP = {
    'car': 0, 'taxi': 0, 'van': 0, 'jeep': 0,
    'motorcycle': 1, 'scooter': 1, 'moped': 1,
    'bus': 2, 'minibus': 2,
    'truck': 3, 'pickup': 3, 'trailer': 3, 'tipper': 3,
    'bicycle': 4,
    'autorickshaw': 0, 'auto-rickshaw': 0, 'e-rickshaw': 0,
}

for split in ('train', 'valid'):
    (OUT_DIR / split / 'images').mkdir(parents=True, exist_ok=True)
    (OUT_DIR / split / 'labels').mkdir(parents=True, exist_ok=True)
MODEL_OUT.mkdir(parents=True, exist_ok=True)

def xyxy2yolo(x1, y1, x2, y2, W, H, cls):
    cx = max(0.001, min(0.999, (x1+x2)/2/W))
    cy = max(0.001, min(0.999, (y1+y2)/2/H))
    w  = max(0.001, min(0.999, (x2-x1)/W))
    h  = max(0.001, min(0.999, (y2-y1)/H))
    return f'{cls} {cx:.6f} {cy:.6f} {w:.6f} {h:.6f}'

print('Config OK — classes:', CLASS_NAMES)

In [ ]:
# ── BDD100K ───────────────────────────────────────────────────────────────────
# Dataset is pre-converted to YOLO format — just copy + verify class IDs

def process_bdd(split):
    out_split = 'valid' if split == 'val' else split
    out_img = OUT_DIR / out_split / 'images'
    out_lbl = OUT_DIR / out_split / 'labels'

    # Try common layout variants
    candidates = [
        (f'{BDD100K_DIR}/{split}/images', f'{BDD100K_DIR}/{split}/labels'),
        (f'{BDD100K_DIR}/images/{split}', f'{BDD100K_DIR}/labels/{split}'),
        (f'{BDD100K_DIR}/{split}',        f'{BDD100K_DIR}/{split}'),
    ]
    src_img = src_lbl = None
    for si, sl in candidates:
        if os.path.isdir(si):
            src_img, src_lbl = si, sl
            break

    if src_img is None:
        print(f'BDD100K {split}: no images dir found — trying JSON fallback')
        return process_bdd_json(split)

    n = 0
    for img_file in os.listdir(src_img):
        if not img_file.lower().endswith(('.jpg', '.jpeg', '.png')):
            continue
        stem = os.path.splitext(img_file)[0]
        lbl_src = os.path.join(src_lbl, stem + '.txt')
        if not os.path.exists(lbl_src):
            continue
        shutil.copy(os.path.join(src_img, img_file), out_img / f'bdd_{img_file}')
        shutil.copy(lbl_src, out_lbl / f'bdd_{stem}.txt')
        n += 1
    print(f'BDD100K {split}: {n} images')
    return n

def process_bdd_json(split):
    out_split = 'valid' if split == 'val' else split
    out_img = OUT_DIR / out_split / 'images'
    out_lbl = OUT_DIR / out_split / 'labels'
    label_file = None
    for p in [
        f'{BDD100K_DIR}/bdd100k/labels/det_20/det_{split}.json',
        f'{BDD100K_DIR}/bdd100k/labels/bdd100k_labels_images_{split}.json',
    ]:
        if os.path.exists(p):
            label_file = p; break
    if not label_file:
        print(f'BDD100K {split}: no annotation file — skipping'); return 0
    with open(label_file) as f:
        anns = json.load(f)
    n = 0
    for ann in anns:
        img_name = ann['name']
        img_src = f"{BDD100K_DIR}/bdd100k/images/100k/{split}/{img_name}"
        if not os.path.exists(img_src): continue
        lines = []
        for lbl in ann.get('labels', []):
            cls = BDD_MAP.get(lbl.get('category', ''))
            b = lbl.get('box2d')
            if cls is None or not b: continue
            if b['x2'] <= b['x1'] or b['y2'] <= b['y1']: continue
            lines.append(xyxy2yolo(b['x1'],b['y1'],b['x2'],b['y2'],1280,720,cls))
        if not lines: continue
        stem = os.path.splitext(img_name)[0]
        shutil.copy(img_src, out_img / f'bdd_{img_name}')
        (out_lbl / f'bdd_{stem}.txt').write_text('\n'.join(lines)+'\n')
        n += 1
    print(f'BDD100K {split} (JSON): {n} images')
    return n

n_bdd_train = process_bdd('train')
n_bdd_val   = process_bdd('val')

In [ ]:
# ── IDD (Pascal VOC XML → YOLO) ───────────────────────────────────────────────

def parse_idd_xml(xml_path):
    root = ET.parse(xml_path).getroot()
    size = root.find('size')
    W = int(size.find('width').text)  if size is not None else 0
    H = int(size.find('height').text) if size is not None else 0
    lines = []
    for obj in root.findall('object'):
        name_el = obj.find('name')
        if name_el is None: continue
        cls = IDD_MAP.get(name_el.text.strip().lower())
        if cls is None: continue
        b = obj.find('bndbox')
        if b is None: continue
        x1,y1,x2,y2 = float(b.find('xmin').text),float(b.find('ymin').text),\
                       float(b.find('xmax').text),float(b.find('ymax').text)
        if x2<=x1 or y2<=y1 or W<=0 or H<=0: continue
        lines.append(xyxy2yolo(x1,y1,x2,y2,W,H,cls))
    return W, H, lines

def process_idd(split):
    out_split = 'valid' if split == 'val' else split
    out_img = OUT_DIR / out_split / 'images'
    out_lbl = OUT_DIR / out_split / 'labels'
    ann_root = os.path.join(IDD_DIR, 'Annotations', split)
    img_root = os.path.join(IDD_DIR, 'JPEGImages', split)
    if not os.path.isdir(ann_root):
        print(f'IDD {split}: Annotations/{split} not found — skipping'); return 0
    n = 0
    for seq in sorted(os.listdir(ann_root)):
        seq_ann = os.path.join(ann_root, seq)
        seq_img = os.path.join(img_root, seq)
        if not os.path.isdir(seq_ann): continue
        for xml_file in os.listdir(seq_ann):
            if not xml_file.endswith('.xml'): continue
            stem = os.path.splitext(xml_file)[0]
            W, H, lines = parse_idd_xml(os.path.join(seq_ann, xml_file))
            if not lines: continue
            img_path = None
            for ext in ('.jpg','.jpeg','.png'):
                c = os.path.join(seq_img, stem+ext)
                if os.path.exists(c): img_path=c; break
            if img_path is None: continue
            if W==0 or H==0:
                img = cv2.imread(img_path)
                if img is None: continue
                H, W = img.shape[:2]
                _, _, lines = parse_idd_xml(os.path.join(seq_ann, xml_file))
                if not lines: continue
            out_stem = f'idd_{seq}_{stem}'
            dst_img = out_img / (out_stem + '.jpg')
            if img_path.endswith(('.jpg','.jpeg')):
                shutil.copy(img_path, dst_img)
            else:
                cv2.imwrite(str(dst_img), cv2.imread(img_path), [cv2.IMWRITE_JPEG_QUALITY,90])
            (out_lbl / (out_stem + '.txt')).write_text('\n'.join(lines)+'\n')
            n += 1
    print(f'IDD {split}: {n} images')
    return n

n_idd_train = process_idd('train')
n_idd_val   = process_idd('val')

print(f'\nDataset summary:')
print(f'  Train: BDD={n_bdd_train} + IDD={n_idd_train} = {n_bdd_train+n_idd_train}')
print(f'  Val:   BDD={n_bdd_val}   + IDD={n_idd_val}   = {n_bdd_val+n_idd_val}')

In [ ]:
# ── Write data.yaml ───────────────────────────────────────────────────────────
yaml_path = OUT_DIR / 'data.yaml'
yaml_path.write_text(f"""path: {OUT_DIR.resolve()}
train: train/images
val:   valid/images

nc: {len(CLASS_NAMES)}
names: {CLASS_NAMES}
""")
print(yaml_path.read_text())

In [ ]:
# ── Train YOLO12x ─────────────────────────────────────────────────────────────
from ultralytics import YOLO

model = YOLO(MODEL)

results = model.train(
    data    = str(yaml_path),
    epochs  = EPOCHS,
    batch   = BATCH,
    imgsz   = IMGSZ,
    device  = '0,1',        # T4 x2; change to '0' for single GPU
    workers = WORKERS,
    project = str(MODEL_OUT),
    name    = 'yolo12x_bdd_idd',
    exist_ok= True,
    # Optimizer
    optimizer     = 'AdamW',
    lr0           = 1e-3,
    lrf           = 0.01,
    weight_decay  = 5e-4,
    warmup_epochs = 3,
    # Augmentation
    hsv_h   = 0.015,
    hsv_s   = 0.7,
    hsv_v   = 0.4,
    degrees = 0.0,          # dashcam is always horizontal
    translate = 0.1,
    scale     = 0.5,
    fliplr    = 0.5,
    mosaic    = 1.0,
    mixup     = 0.1,
    # Callbacks
    patience    = 15,
    save        = True,
    save_period = 5,
    val         = True,
    plots       = True,
    verbose     = True,
)

In [ ]:
# ── Evaluate best checkpoint ──────────────────────────────────────────────────
from ultralytics import YOLO
import pandas as pd

best_pt = MODEL_OUT / 'yolo12x_bdd_idd' / 'weights' / 'best.pt'
model   = YOLO(str(best_pt))

metrics = model.val(data=str(yaml_path), imgsz=IMGSZ, verbose=True)

print(f'\nmAP50:    {metrics.box.map50:.4f}')
print(f'mAP50-95: {metrics.box.map:.4f}')
print(f'Precision:{metrics.box.mp:.4f}')
print(f'Recall:   {metrics.box.mr:.4f}')

# Per-class AP
if hasattr(metrics.box, 'ap_class_index'):
    for i, cls_idx in enumerate(metrics.box.ap_class_index):
        print(f'  {CLASS_NAMES[cls_idx]:12s} AP50={metrics.box.ap50[i]:.3f}')

In [ ]:
# ── Save final weights ────────────────────────────────────────────────────────
import shutil

best_pt  = MODEL_OUT / 'yolo12x_bdd_idd' / 'weights' / 'best.pt'
final_pt = Path('/kaggle/working/argus_yolo12x_best.pt')
shutil.copy(best_pt, final_pt)
print(f'Saved: {final_pt}  ({final_pt.stat().st_size/1e6:.1f} MB)')
print('\nDownload this file and use with:')
print('  VehicleDetector(model_path="argus_yolo12x_best.pt")')

# Also export to ONNX for Jetson TensorRT
model.export(format='onnx', imgsz=IMGSZ, simplify=True)
print('ONNX exported for Jetson TensorRT pipeline.')